In [1]:
import importlib
import sys

# performance imports for torch: torch kernel uses one core only.
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TORCH_NUM_THREADS"] = "1" 

import torch

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')
sys.path.insert(0, '../../../..')
sys.path.insert(0, '../../../../..')
sys.path.insert(0, '../../../../../..')

In [2]:
# Load decision-labeled data
file_path_train = '../../../../../../data/BPIC20_DD/tensor_data/decision_labeled/bpic20_dd_all_5_train.pkl'
bpic20_dd_train_dataset = torch.load(file_path_train, weights_only=False)

file_path_val = '../../../../../../data/BPIC20_DD/tensor_data/decision_labeled/bpic20_dd_all_5_val.pkl'
bpic20_dd_val_dataset = torch.load(file_path_val, weights_only=False)

In [3]:
# BPIC20_DD dataset dynamic categories, features:
bpic20_dd_all_categories = bpic20_dd_train_dataset.all_categories

bpic20_dd_all_categories_cat = bpic20_dd_all_categories[0]
# print(bpic20_dd_all_categories_cat)
bpic20_dd_all_categories_num = bpic20_dd_all_categories[1]
# print(bpic20_dd_all_categories_num)
for i, cat in enumerate(bpic20_dd_all_categories_cat):
     print(f"BPIC20_DD dynamic categorical feature: {cat[0]}, Index position in categorical data list: {i}")
     print(f"BPIC20_DD amount of category labels: {cat[1]}")
print('\n')    
for i, num in enumerate(bpic20_dd_all_categories_num):
     print(f"BPIC20_DD dynamic numerical feature: {num[0]}, Index position in categorical data list: {i}")
     print(f"BPIC20_DD amount of numerical: {num[1]}")
print('\n')
     
# BPIC20_DD dataset static categories, features:
bpic20_dd_all_stat_categories = bpic20_dd_train_dataset.all_static_categories

bpic20_dd_all_stat_categories_cat = bpic20_dd_all_stat_categories[0]
# print(bpic20_dd_all_stat_categories_cat)
bpic20_dd_all_stat_categories_num = bpic20_dd_all_stat_categories[1]
# print(bpic20_dd_all_stat_categories_num)
for i, cat in enumerate(bpic20_dd_all_stat_categories_cat):
     print(f"BPIC20_DD static categorical feature: {cat[0]}, Index position in categorical data list: {i}")
     print(f"BPIC20_DD amount of category labels: {cat[1]}")
print('\n')    
for i, num in enumerate(bpic20_dd_all_stat_categories_num):
     print(f"BPIC20_DD static numerical feature: {num[0]}, Index position in categorical data list: {i}")
     print(f"BPIC20_DD amount of numerical: {num[1]}")  

BPIC20_DD dynamic categorical feature: concept:name, Index position in categorical data list: 0
BPIC20_DD amount of category labels: 18
BPIC20_DD dynamic categorical feature: org:resource, Index position in categorical data list: 1
BPIC20_DD amount of category labels: 22
BPIC20_DD dynamic categorical feature: budget_status, Index position in categorical data list: 2
BPIC20_DD amount of category labels: 4
BPIC20_DD dynamic categorical feature: supplier_type, Index position in categorical data list: 3
BPIC20_DD amount of category labels: 6
BPIC20_DD dynamic categorical feature: goods_match, Index position in categorical data list: 4
BPIC20_DD amount of category labels: 5


BPIC20_DD dynamic numerical feature: case_elapsed_time, Index position in categorical data list: 0
BPIC20_DD amount of numerical: 1
BPIC20_DD dynamic numerical feature: event_elapsed_time, Index position in categorical data list: 1
BPIC20_DD amount of numerical: 1
BPIC20_DD dynamic numerical feature: day_in_week, Index

In [4]:
# Create lists with name of Encoder features (input) and decoder features (input & output)

# Encoder features (fixed): use only requested features
enc_feat_cat = ['concept:name', 'org:resource']
enc_feat_num = ['case_elapsed_time']
enc_feat = [enc_feat_cat, enc_feat_num]
print("Input features encoder: ", enc_feat)

# Decoder features (unused by C-LSTM training cell, kept for consistency)
dec_feat_cat = ['concept:name']
dec_feat_num = []
dec_feat = [dec_feat_cat, dec_feat_num]
print("Features decoder: ", dec_feat)

# Guard tensors are pre-computed and stored in the dataset .pkl files
# (prepared during data loading via prepare_guard_tensors).
print(f"Guard tensors: targets {bpic20_dd_train_dataset._guard_targets.shape}, "
      f"mask {bpic20_dd_train_dataset._guard_mask.shape}, "
      # is that still present?
      f"confidence {bpic20_dd_train_dataset._guard_confidence.shape}")

Input features encoder:  [['concept:name', 'org:resource'], ['case_elapsed_time']]
Features decoder:  [['concept:name'], []]
Guard tensors: targets torch.Size([83492, 29, 18]), mask torch.Size([83492, 29]), confidence torch.Size([83492, 29])


In [5]:
import suffix_pred.models.C_LSTM
importlib.reload(suffix_pred.models.C_LSTM)
from suffix_pred.models.C_LSTM import FullShared_Join_LSTM

# Size hidden layer
hidden_size = 50

# Number of LSTM cells
num_layers = 1

# STANDARD: One numerical output to predict
input_size = 1

# C-LSTM uses dynamic prefix features only
model_feat = enc_feat

# Output size: activity classes only
activity_feature_name = 'concept:name'
activity_feature_index = next(i for i, cat in enumerate(bpic20_dd_all_categories_cat) if cat[0] == activity_feature_name)
output_size_act = bpic20_dd_all_categories_cat[activity_feature_index][1]

# C-LSTM model initialization
model = FullShared_Join_LSTM(data_set_categories=bpic20_dd_all_categories,
                             hidden_size=hidden_size,
                             num_layers=num_layers,
                             model_feat=model_feat,
                             input_size=input_size,
                             output_size_act=output_size_act)

Data set categories:  ([('concept:name', 18, {'Approve Invoice': 1, 'Approve Requisition': 2, 'Close Case': 3, 'Collect Quotations': 4, 'Create Purchase Order': 5, 'Create Purchase Requisition': 6, 'EOS': 7, 'Evaluate Quotations': 8, 'Flag Invoice Mismatch': 9, 'Pay Invoice': 10, 'Receive Goods': 11, 'Reject Requisition': 12, 'Reorder Goods': 13, 'Request Credit Note': 14, 'Revise Requisition': 15, 'Select Supplier': 16, 'Send Purchase Order': 17}), ('org:resource', 22, {'Alice': 1, 'Bob': 2, 'Buyer_1': 3, 'Buyer_2': 4, 'Buyer_3': 5, 'Carol': 6, 'Clerk_1': 7, 'Clerk_2': 8, 'Clerk_3': 9, 'David': 10, 'EOS': 11, 'Eva': 12, 'Frank': 13, 'Manager_FIN_1': 14, 'Manager_FIN_2': 15, 'Manager_IT_1': 16, 'Manager_IT_2': 17, 'Manager_OPS_1': 18, 'Manager_OPS_2': 19, 'Receiver_A': 20, 'Receiver_B': 21}), ('budget_status', 4, {'EOS': 1, 'approved': 2, 'pending': 3}), ('supplier_type', 6, {'EOS': 1, 'preferred': 2, 'risky': 3, 'standard': 4, nan: 5}), ('goods_match', 5, {'EOS': 1, 'False': 2, 'True'

/home/PSPLab/.local/share/virtualenvs/decision_aware_augmentation_for_PPM-0DzgvVpC/lib/python3.12/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


In [6]:
import suffix_pred.loss
importlib.reload(suffix_pred.loss)
from suffix_pred.loss import Loss

loss_obj = Loss()

In [7]:
import suffix_pred.train
importlib.reload(suffix_pred.train)
from suffix_pred.train import CTraining

from torch.optim.lr_scheduler import ReduceLROnPlateau

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Start learning rate
learning_rate = 0.001
weight_decay = 0.0

# Optimizer and Scheduler
optimizer = torch.optim.Adam(params=model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=15, min_lr=1e-8)


# Keep consistent across all models
# Epochs / Batch size
num_epochs = 100
batch_size = 128

# Shuffle data
shuffle = True

optimize_values = {"optimizer": optimizer,
                   "scheduler": scheduler,
                   "epochs": num_epochs,
                   "mini_batches": batch_size,
                   "shuffle": shuffle,
                   # Optional the higher the smaller get S:
                   "guard_support_threshold": 0.0}

# Activity feature index and EOS id for activity-only next-event prediction
activity_feature_name = 'concept:name'
concept_name_id = next(i for i, cat in enumerate(bpic20_dd_all_categories_cat) if cat[0] == activity_feature_name)
activity_label_to_id = bpic20_dd_all_categories_cat[concept_name_id][2]
eos_id = activity_label_to_id.get('EOS')
if eos_id is None:
    raise ValueError("Could not find EOS id in activity label mapping.")

# Decision-aware guard regularization weight (λ_g)
lambda_g = 1.0
print(f"Decision-aware training: λ_g = {lambda_g}")

model_output_path = "../../../../../../models/BPIC20_DD/decision/BPIC20_DD_C_LSTM_v1_DA.pkl"
os.makedirs(os.path.dirname(model_output_path), exist_ok=True)

trainer = CTraining(device=device,
                    model=model,
                    data_train=bpic20_dd_train_dataset,
                    data_val=bpic20_dd_val_dataset,
                    optimize_values=optimize_values,
                    concept_name_id=concept_name_id,
                    eos_id=eos_id,
                    loss_obj=loss_obj,
                    lambda_g=lambda_g,
                    save_model_n_th_epoch=1,
                    saving_path=model_output_path)

# Train the model (decision-aware: L_base + λ_g * L_guard)
trainer.train()

Device: cuda
Decision-aware training: λ_g = 1.0
Device:  cuda
Optimizer:  Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0.0
)
Scheduler:  <torch.optim.lr_scheduler.ReduceLROnPlateau object at 0x7f3fa9350830>
Epochs:  100
Mini baches:  128
Shuffle batched dataset:  True


  0%|          | 0/100 [00:00<?, ?it/s]

Epoch [1/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 0.9921
Validation: Avg Validation Loss: 0.1355
Validation Loss for Scheduler: 0.1355
saving model
Epoch [2/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 0.5225
Validation: Avg Validation Loss: 0.1271
Validation Loss for Scheduler: 0.1271
saving model
Epoch [3/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 0.5115
Validation: Avg Validation Loss: 0.1240
Validation Loss for Scheduler: 0.1240
saving model
Epoch [4/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 0.5100
Validation: Avg Validation Loss: 0.1229
Validation Loss for Scheduler: 0.1229
saving model
Epoch [5/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 0.5069
Validation: Avg Validation Loss: 0.1227
Validation Loss for Scheduler: 0.1227
saving model
Epoch [6/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 0.5056
Validation: Avg Validation Loss: 0.1248
Validat

([0.992053788914067,
  0.5225180413283032,
  0.5115415729342705,
  0.5100263876307065,
  0.506930869009922,
  0.505605499900501,
  0.5088920348636224,
  0.5090394693257068,
  0.5059120272604284,
  0.5020451075176368,
  0.504419137311566,
  0.5049377804191335,
  0.5030058742619218,
  0.5041486695979309,
  0.503969280314482,
  0.5027972203529629,
  0.5000685031253382,
  0.5006348893251755,
  0.5010979686725376,
  0.499960390956442,
  0.5010382786598542,
  0.5011636283231549,
  0.5004550383111054,
  0.49995825744332434,
  0.498104214166244,
  0.5005007301775274,
  0.49766598470725837,
  0.49799369058784265,
  0.49172582156716593,
  0.49095175854553674,
  0.48866226306187976,
  0.48822499179547996,
  0.4875396860001828,
  0.48902037897208567,
  0.48836441735122327,
  0.4893681587020982,
  0.4872840271655122,
  0.4891554525900763,
  0.4879239363792662,
  0.48577345798154337,
  0.48648211734218216,
  0.4862901171466841,
  0.48575724556380356,
  0.488742436329402,
  0.4855091811641252,
  0.48